# 01 — Image Basics with PyTorch and torchvision

Computer vision = teaching machines to understand images.

You'll learn:
- How images are represented as tensors
- How to load, transform, and visualize images
- How to use a **pre-trained model** (ResNet) for classification — no training needed!
- How to interpret the model's predictions

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import urllib.request
import io

device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')
print(f'torchvision version: {torchvision.__version__}')

## Part 1: Images as Tensors

A color image is a **3D tensor**: `[channels, height, width]`  
- Channels = 3 (Red, Green, Blue)
- Each pixel value: 0–255 (uint8) or 0.0–1.0 (float)

In [ ]:
# Create a tiny synthetic 'image' from scratch
# Shape: [3 channels, 4 rows, 4 cols]
tiny_image = torch.zeros(3, 4, 4)

tiny_image[0, :, :] = 1.0  # Red channel = full brightness
tiny_image[1, :, :] = 0.0  # Green channel = off
tiny_image[2, :, :] = 0.0  # Blue channel = off

print('Image tensor shape (C, H, W):', tiny_image.shape)

# Convert to displayable format: (H, W, C) with values 0-1
display = tiny_image.permute(1, 2, 0).numpy()
plt.figure(figsize=(2, 2))
plt.imshow(display)
plt.title('Red square')
plt.axis('off')
plt.show()

In [ ]:
# Make a gradient image
gradient = torch.zeros(3, 64, 64)
for i in range(64):
    gradient[0, :, i] = i / 63  # red increases left to right
    gradient[2, i, :] = i / 63  # blue increases top to bottom

plt.figure(figsize=(3, 3))
plt.imshow(gradient.permute(1, 2, 0).numpy())
plt.title('RGB Gradient')
plt.axis('off')
plt.show()

## Part 2: Image Transforms

Before feeding images to a model, we standardize them (resize, normalize). `torchvision.transforms` makes this easy.

In [ ]:
# The standard transform for ImageNet-pretrained models
preprocess = transforms.Compose([
    transforms.Resize(256),            # resize shortest side to 256px
    transforms.CenterCrop(224),        # crop center 224x224 (ResNet input size)
    transforms.ToTensor(),             # PIL Image → tensor [0, 1]
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],    # ImageNet mean
        std=[0.229, 0.224, 0.225]      # ImageNet std
    ),
])

print('Transform pipeline defined!')

# Show what each step does
print()
print('Pipeline steps:')
for i, t in enumerate(preprocess.transforms):
    print(f'  {i+1}. {t}')

## Part 3: CIFAR-10 Dataset

CIFAR-10 is a classic benchmark: 60,000 tiny (32x32) color images across 10 categories. Great for learning without needing a big GPU.

In [ ]:
# Download CIFAR-10 (about 170MB, cached after first download)
cifar_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),  # scale to [-1, 1]
])

trainset = torchvision.datasets.CIFAR10(
    root='../../data/cifar10', train=True, download=True, transform=cifar_transform
)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
print(f'Training images: {len(trainset)}')
print(f'Classes: {classes}')

In [ ]:
# Visualize a batch of images
loader = torch.utils.data.DataLoader(trainset, batch_size=16, shuffle=True)
images, labels = next(iter(loader))

# Un-normalize for display
def imshow(img):
    img = img / 2 + 0.5  # undo normalize
    return img.permute(1, 2, 0).numpy()

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(imshow(images[i]))
    ax.set_title(classes[labels[i]], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample CIFAR-10 Images', y=1.02)
plt.tight_layout()
plt.show()

## Part 4: Transfer Learning with a Pretrained Model

Instead of training from scratch (which takes days/GPUs), we use a model already trained on 1 million+ images.

We'll use **ResNet-18** trained on ImageNet (1000 classes) to classify CIFAR-10 images.

In [ ]:
# Load pretrained ResNet-18
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Replace the final layer: 1000 ImageNet classes → 10 CIFAR-10 classes
num_features = resnet.fc.in_features
resnet.fc = torch.nn.Linear(num_features, 10)

resnet = resnet.to(device)
print('Model ready!')
print(f'Final layer: {resnet.fc}')

In [ ]:
# Quick test: pass one batch through the model (no training yet)
resnet.eval()

# CIFAR-10 images are 32x32, but ResNet expects 224x224
# Use a separate transform that resizes
resize_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

testset_resized = torchvision.datasets.CIFAR10(
    root='../../data/cifar10', train=False, download=True, transform=resize_transform
)
test_loader = torch.utils.data.DataLoader(testset_resized, batch_size=32, shuffle=False)

images, labels = next(iter(test_loader))
images = images.to(device)

with torch.no_grad():
    outputs = resnet(images)
    _, predicted = torch.max(outputs, 1)

# Show first 8 predictions (without fine-tuning, expect ~random accuracy)
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i, ax in enumerate(axes):
    ax.imshow(images[i].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5)
    color = 'green' if predicted[i] == labels[i] else 'red'
    ax.set_title(f'Pred: {classes[predicted[i]]}\nTrue: {classes[labels[i]]}',
                 fontsize=7, color=color)
    ax.axis('off')
plt.tight_layout()
plt.show()

print('\nGreen = correct, Red = wrong')
print('Without fine-tuning, accuracy is random — the final layer weights are random!')

## What's Next?

To actually fine-tune this model on CIFAR-10, you'd run a training loop (like in the PyTorch intro notebook) for a few epochs. With just the final layer training, you'd get ~70-80% accuracy in ~10 minutes on CPU.

**Try:** Adapt the training loop from `02_deep_learning/01_pytorch_intro.ipynb` to fine-tune the ResNet here!

**Next:** `04_local_llms/01_ollama_chat.ipynb` — talk to a local language model!